# Лабораторная работа 1. Вариант 5 — Криптовалютная биржа (High-Frequency Trading)
## Выполнил студент группы ИУ5-85Б Андрест Владислав

---

## **Содержание**

### **ЧАСТЬ 1. Парето-оптимизация конфигураций серверов**
1. Постановка задачи
2. Загрузка данных
3. Фильтрация по требованиям HFT
4. Парето-оптимизация
5. Нормирование критериев
6. Идеальная точка и расстояние
7. Линейная свертка с приоритетами

### **ЧАСТЬ 2. Масштабирование инфраструктуры**
8. Определение весовых коэффициентов для сегментов
9. Расчет критериев предпочтения для каждого сегмента
10. Требования к сетевой инфраструктуре (10–25G RDMA, 10G для DB)
11. Требования к низкой задержке и ускорению
12. Двухуровневая оптимизация
13. Визуализация результатов
14. Выводы

---
# **ЧАСТЬ 1. ПАРЕТО-ОПТИМИЗАЦИЯ СЕРВЕРНЫХ КОНФИГУРАЦИЙ (HFT)**
---

## 1. Постановка задачи (Часть 1)

Требуется подобрать конфигурации серверов для развёртывания инфраструктуры **криптовалютной биржи (High-Frequency Trading, HFT)**.

**Требования к системе:**

1. **Микросекундные задержки** — минимальное время отклика.
2. **Аппаратное ускорение сетевых операций** — высокопроизводительные сетевые интерфейсы.
3. **Максимальная отказоустойчивость** — резервирование и надёжные компоненты.
4. **Полное резервирование инфраструктуры** — дублирование критических узлов.

**Количественные ограничения:**

- CPU_Cores ≥ 16 (высокая частота и многопоточность).
- RAM_GB ≥ 64 (обработка большого количества ордеров).
- Storage_Type ∈ {SSD, NVMe} (минимальная задержка I/O).
- Network_Speed_Gbps ≥ 10 (низкая сетевая задержка).
- Condition ∈ {Like New, New} (минимизация риска отказов).
- Power_Consumption_W ≤ 800.
- Price_USD ≤ 8000.

---
## 2. Загрузка данных

In [ ]:
# Загрузка необходимых библиотек
if (!require(readxl)) install.packages("readxl", repos = "https://cloud.r-project.org")
if (!require(dplyr)) install.packages("dplyr", repos = "https://cloud.r-project.org")
if (!require(ggplot2)) install.packages("ggplot2", repos = "https://cloud.r-project.org")
if (!require(tidyr)) install.packages("tidyr", repos = "https://cloud.r-project.org")

library(readxl)
library(dplyr)
library(ggplot2)
library(tidyr)

# Загрузка данных из CSV файла
Data <- read.csv("https://raw.githubusercontent.com/junaart/AHP_KIS/refs/heads/main/Lab_1/computer_dataset_1000.csv", sep = ",")
View(Data)
colnames(Data)
cat("Размер датасета:", nrow(Data), "x", ncol(Data), "\n")

# Приводим значения скорости Ethernet к числовым значениям в Gbps
ethernet_speed_value <- as.numeric(gsub("[^0-9.]", "", Data$Ethernet_Speed))

is_gbps <- grepl("gbps", Data$Ethernet_Speed, ignore.case = TRUE)
is_mbps <- grepl("mbps", Data$Ethernet_Speed, ignore.case = TRUE)

# Если явно Gbps — оставляем как есть
# Если явно Mbps или единицы не указаны — считаем что это Mbps и переводим в Gbps
Data$Ethernet_Speed_Gbps <- ifelse(is_gbps, ethernet_speed_value,
                            ifelse(is_mbps, ethernet_speed_value / 1000,
                                   ethernet_speed_value / 1000))

Data$Ethernet_Speed_Gbps <- as.numeric(Data$Ethernet_Speed_Gbps)
summary(Data$Ethernet_Speed_Gbps)

ID,CPU_Frequency_GHz,CPU_Cores,CPU_Brand,CPU_Generation,RAM_GB,Storage_GB,Storage_Type,GPU_Type,GPU_Model,⋯,Brand,Model,Purchase_Date,Age_Days,Condition,Price_USD,Power_Consumption_W,Weight_kg,Screen_Size_inch,Case_Type
<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
PC-0001,2.64,4,Intel Core i9,11,16,1000,SSD,Dedicated,AMD Radeon RX 6600,⋯,Apple,Inspiron 1409,2023-07-29,930,Good,1159,238,NA,NA,Full Tower
PC-0002,5.00,8,AMD Ryzen 3,8,16,2000,NVMe,Integrated,Intel Iris Xe,⋯,Apple,ZBook 5012,2023-09-14,883,Good,1616,218,NA,NA,
PC-0003,4.10,12,AMD Ryzen 9,12,256,512,NVMe,Dedicated,NVIDIA RTX 3060,⋯,Lenovo,EliteBook 3286,2025-06-06,252,Like New,2568,393,3.1,13.3,
PC-0004,3.55,10,Intel Core i7,11,16,1000,HDD,Integrated,Intel UHD,⋯,Samsung,Inspiron 9935,2025-10-01,135,Like New,1619,118,NA,27.0,
PC-0005,1.74,12,AMD Ryzen 9,8,16,1000,HDD,Integrated,Intel UHD,⋯,Dell,Inspiron 7912,2024-09-08,523,New,1570,128,NA,NA,Small Form Factor
PC-0006,1.74,8,Intel Core i3,14,128,2000,HDD,Dedicated,NVIDIA RTX 4080,⋯,HP,XPS 1488,2023-12-02,804,Good,2285,324,NA,16.0,
PC-0007,1.34,8,AMD Ryzen 7,10,32,4000,SSD,Dedicated,NVIDIA RTX 3060,⋯,Dell,Inspiron 4582,2024-04-24,660,Good,2093,296,NA,NA,Full Tower
PC-0008,4.65,12,AMD Ryzen 9,7,128,128,SSD,Dedicated,NVIDIA RTX 3060,⋯,Lenovo,EliteBook 9279,2025-03-31,NA,Fair,2445,325,1.0,14.0,
PC-0009,3.56,4,Intel Core i5,11,8,1000,SSD,Integrated,Intel UHD,⋯,HP,XPS 4257,2025-11-25,80,Fair,1156,84,3.0,13.3,


[1] "ID"                  "CPU_Frequency_GHz"   "CPU_Cores"          
 [4] "CPU_Brand"           "CPU_Generation"      "RAM_GB"             
 [7] "Storage_GB"          "Storage_Type"        "GPU_Type"           
[10] "GPU_Model"           "GPU_Memory_GB"       "WIFI"               
[13] "WIFI_Standard"       "Bluetooth"           "Bluetooth_Version"  
[16] "Computer_Type"       "Operating_System"    "USB_3_Ports"        
[19] "USB_C_Ports"         "USB_2_Ports"         "Ethernet_Speed"     
[22] "Brand"               "Model"               "Purchase_Date"      
[25] "Age_Days"            "Condition"           "Price_USD"          
[28] "Power_Consumption_W" "Weight_kg"           "Screen_Size_inch"   
[31] "Case_Type"

Размер датасета: 1000 x 31 


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
  0.100   1.000   1.000   2.143   2.500  10.000      46 

---
## 3. Фильтрация по требованиям криптовалютной биржи

In [ ]:
# Отбор по критериям варианта 5 (Криптовалютная биржа, HFT)
C <- Data$ID[
  (Data$CPU_Cores >= 16) &
  (Data$RAM_GB >= 64) &
  (Data$Storage_GB >= 1000) &
  (Data$Ethernet_Speed_Gbps  >= 2.5) &
  (Data$Price_USD <= 8000) &
  (Data$Storage_Type %in% c("SSD", "NVMe")) &
  (Data$Condition %in% c("Like New", "New")) &
  (!is.na(Data$Power_Consumption_W)) &
  (Data$Power_Consumption_W <= 800)
]

C

[1] "PC-0045" "PC-0075" "PC-0186" "PC-0231" "PC-0325" "PC-0956"

In [ ]:
# Убираем пропущенные значения
C <- C[!is.na(C)]
cat("Количество конфигураций, прошедших отбор:", length(C), "\n")
C

Количество конфигураций, прошедших отбор: 6 


[1] "PC-0045" "PC-0075" "PC-0186" "PC-0231" "PC-0325" "PC-0956"

---
## 4. Формирование датафрейма для Парето-оптимизации

In [ ]:
F <- data.frame(
  Data$CPU_Cores[Data$ID %in% C],
  Data$RAM_GB[Data$ID %in% C],
  Data$Storage_GB[Data$ID %in% C],
  Data$Ethernet_Speed_Gbps[Data$ID %in% C],
  1 / Data$Price_USD[Data$ID %in% C],
  1 / Data$Power_Consumption_W[Data$ID %in% C]
)

colnames(F) <- c(
  "CPU_Cores",
  "RAM_GB",
  "Storage_GB",
  "Ethernet_Speed_Gbps",
  "Inv_Price_USD",
  "Inv_Power_W"
)

rownames(F) <- C

View(F)

,CPU_Cores,RAM_GB,Storage_GB,Ethernet_Speed_Gbps,Inv_Price_USD,Inv_Power_W
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PC-0045,16,256,2000,2.5,0.0003065604,0.002336449
PC-0075,32,256,1000,2.5,0.0002319647,0.002008032
PC-0186,24,512,1000,2.5,0.0001990050,0.002293578
PC-0231,16,64,1000,2.5,0.0004576659,0.005813953
PC-0325,32,128,2000,10.0,0.0002674512,0.002252252
PC-0956,24,64,4000,2.5,0.0003051572,0.004132231


---
## 5. Парето-оптимизация

In [ ]:
# Проверка: X доминирует по Парето Y (X >_p Y)
DPareto <- function(X, Y) {
  p <- TRUE
  l <- FALSE
  i <- 1
  while (p & (i <= length(X))) {
    if (X[i] < Y[i]) p <- FALSE
    if (X[i] > Y[i]) l <- TRUE
    i <- i + 1
  }
  if (!p | !l) return(FALSE)
  else return(TRUE)
}

result <- c()
for (i in c(1:length(rownames(F)))) {
  p <- TRUE
  for (j in c(1:length(rownames(F)))) {
    if (DPareto(F[j, ], F[i, ]))
      p <- FALSE
  }
  if (p) result <- c(result, rownames(F)[i])
}
cat("Парето-оптимальные конфигурации:", length(result), "\n")
result

Парето-оптимальные конфигурации: 6 


[1] "PC-0045" "PC-0075" "PC-0186" "PC-0231" "PC-0325" "PC-0956"

In [ ]:
FF <- data.frame(
  Data$CPU_Cores[Data$ID %in% result],
  Data$RAM_GB[Data$ID %in% result],
  Data$Storage_GB[Data$ID %in% result],
  Data$Ethernet_Speed_Gbps[Data$ID %in% result],
  Data$Price_USD[Data$ID %in% result],
  Data$Power_Consumption_W[Data$ID %in% result]
)

colnames(FF) <- c(
  "CPU_Cores",
  "RAM_GB",
  "Storage_GB",
  "Ethernet_Speed_Gbps",
  "Price_USD",
  "Power_Consumption_W"
)

rownames(FF) <- result
FF

,CPU_Cores,RAM_GB,Storage_GB,Ethernet_Speed_Gbps,Price_USD,Power_Consumption_W
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PC-0045,16,256,2000,2.5,3262,428
PC-0075,32,256,1000,2.5,4311,498
PC-0186,24,512,1000,2.5,5025,436
PC-0231,16,64,1000,2.5,2185,172
PC-0325,32,128,2000,10.0,3739,444
PC-0956,24,64,4000,2.5,3277,242


---
## 6. Нормирование критериев

In [ ]:
# CPU
mmax <- max(FF$CPU_Cores)
mmin <- min(FF$CPU_Cores)
CPU_Cores_norm <- (FF$CPU_Cores - mmin) * 100 / (mmax - mmin)

# RAM
mmax <- max(FF$RAM_GB)
mmin <- min(FF$RAM_GB)
RAM_GB_norm <- (FF$RAM_GB - mmin) * 100 / (mmax - mmin)

# Storage
mmax <- max(FF$Storage_GB)
mmin <- min(FF$Storage_GB)
Storage_GB_norm <- (FF$Storage_GB - mmin) * 100 / (mmax - mmin)

# Network (чем больше — тем лучше)
mmax <- max(FF$Ethernet_Speed_Gbps)
mmin <- min(FF$Ethernet_Speed_Gbps)
Ethernet_Speed_Gbps_norm <- (FF$Ethernet_Speed_Gbps - mmin) * 100 / (mmax - mmin)

# Цена (чем меньше — тем лучше)
mmax <- max(FF$Price_USD)
mmin <- min(FF$Price_USD)
Price_USD_norm <- (mmax - FF$Price_USD) * 100 / (mmax - mmin)

# Энергопотребление (чем меньше — тем лучше)
mmax <- max(FF$Power_Consumption_W)
mmin <- min(FF$Power_Consumption_W)
Power_Consumption_W_norm <- (mmax - FF$Power_Consumption_W) * 100 / (mmax - mmin)

WW <- data.frame(
  CPU_Cores_norm,
  RAM_GB_norm,
  Storage_GB_norm,
  Ethernet_Speed_Gbps_norm,
  Price_USD_norm,
  Power_Consumption_W_norm
)

rownames(WW) <- rownames(FF)

View(WW)

,CPU_Cores_norm,RAM_GB_norm,Storage_GB_norm,Ethernet_Speed_Gbps_norm,Price_USD_norm,Power_Consumption_W_norm
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PC-0045,0,42.85714,33.33333,0,62.07746,21.47239
PC-0075,100,42.85714,0.00000,0,25.14085,0.00000
PC-0186,50,100.00000,0.00000,0,0.00000,19.01840
PC-0231,0,0.00000,0.00000,0,100.00000,100.00000
PC-0325,100,14.28571,33.33333,100,45.28169,16.56442
PC-0956,50,0.00000,100.00000,0,61.54930,78.52761


---
## 7. Идеальная точка и расстояние (евклидова метрика)

In [ ]:
ideal <- apply(WW, 2, max)
ideal

stopifnot(ncol(WW) == length(ideal))
summary(WW)

colnames(WW)

CPU_Cores_norm              RAM_GB_norm          Storage_GB_norm 
                     100                      100                      100 
Ethernet_Speed_Gbps_norm           Price_USD_norm Power_Consumption_W_norm 
                     100                      100                      100

 CPU_Cores_norm   RAM_GB_norm      Storage_GB_norm  Ethernet_Speed_Gbps_norm
 Min.   :  0.0   Min.   :  0.000   Min.   :  0.00   Min.   :  0.00          
 1st Qu.: 12.5   1st Qu.:  3.571   1st Qu.:  0.00   1st Qu.:  0.00          
 Median : 50.0   Median : 28.571   Median : 16.67   Median :  0.00          
 Mean   : 50.0   Mean   : 33.333   Mean   : 27.78   Mean   : 16.67          
 3rd Qu.: 87.5   3rd Qu.: 42.857   3rd Qu.: 33.33   3rd Qu.:  0.00          
 Max.   :100.0   Max.   :100.000   Max.   :100.00   Max.   :100.00          
 Price_USD_norm   Power_Consumption_W_norm
 Min.   :  0.00   Min.   :  0.00          
 1st Qu.: 30.18   1st Qu.: 17.18          
 Median : 53.42   Median : 20.25          
 Mean   : 49.01   Mean   : 39.26          
 3rd Qu.: 61.95   3rd Qu.: 64.26          
 Max.   :100.00   Max.   :100.00          

[1] "CPU_Cores_norm"           "RAM_GB_norm"             
[3] "Storage_GB_norm"          "Ethernet_Speed_Gbps_norm"
[5] "Price_USD_norm"           "Power_Consumption_W_norm"

In [ ]:
distance <- function(A, B) {
  return(sqrt(sum((A - B)^2)))
}

distances <- sapply(result, function(i) distance(ideal, WW[i, ]))
names(distances) <- result
for (i in result) {
  print(paste(i, ":", as.character(distance(ideal, WW[i, ]))))
}

[1] "PC-0045 : 187.921404746074"
[1] "PC-0075 : 197.152730641042"
[1] "PC-0186 : 197.631016653886"
[1] "PC-0231 : 200"
[1] "PC-0325 : 147.468549645876"
[1] "PC-0956 : 156.331443737425"


In [ ]:
# Конфигурация, ближайшая к идеальной точке
best_id <- result[which.min(distances)]
best_distance <- min(distances)
cat("Рекомендуемая конфигурация (по расстоянию до идеальной точки):", best_id, "\n")
cat("Расстояние до идеальной точки:", round(best_distance, 4), "\n\n")
Data[Data$ID == best_id, ]

Рекомендуемая конфигурация (по расстоянию до идеальной точки): PC-0325 
Расстояние до идеальной точки: 147.4685 



,ID,CPU_Frequency_GHz,CPU_Cores,CPU_Brand,CPU_Generation,RAM_GB,Storage_GB,Storage_Type,GPU_Type,GPU_Model,⋯,Model,Purchase_Date,Age_Days,Condition,Price_USD,Power_Consumption_W,Weight_kg,Screen_Size_inch,Case_Type,Ethernet_Speed_Gbps
,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
325,PC-0325,5.01,32,Intel Core i3,7,128,2000,NVMe,Dedicated,AMD Radeon RX 7900,⋯,MacBook 3979,2022-04-06,1409,Like New,3739,444,NA,14,,10


---
## 8. Линейная свертка с приоритетами

In [ ]:
# Порядок критериев: CPU, RAM, Storage, Network, Price, Power

a <- c(3, 3, 2, 3, 2, 3)
anorm <- a / sum(a)

cat("Нормированные веса критериев:\n")
print(anorm)
cat("Сумма весов:", sum(anorm), "\n\n")

stopifnot(length(a) == ncol(WW))

scores <- sapply(result, function(i) sum(anorm * WW[i, ]))
names(scores) <- result

cat("Значения линейной свёртки для Парето-оптимальных альтернатив:\n")
for (i in result) {
  cat(i, ":", round(scores[i], 4), "\n")
}

scores <- sort(scores, decreasing = TRUE)

cat("\nОтсортированные оценки:\n")
print(round(scores, 4))

Нормированные веса критериев:
[1] 0.1875 0.1875 0.1250 0.1875 0.1250 0.1875
Сумма весов: 1 

Значения линейной свёртки для Парето-оптимальных альтернатив:
PC-0045 : 23.9881 
PC-0075 : 29.9283 
PC-0186 : 31.691 
PC-0231 : 31.25 
PC-0325 : 53.1113 
PC-0956 : 44.2926 

Отсортированные оценки:
PC-0325 PC-0956 PC-0186 PC-0231 PC-0075 PC-0045 
53.1113 44.2926 31.6910 31.2500 29.9283 23.9881 


In [ ]:
# Лучшая конфигурация по линейной свёртке
best_weighted_id <- names(scores)[which.max(scores)]
best_weighted_score <- max(scores)

cat("\nРекомендуемая конфигурация (по линейной свёртке):", best_weighted_id, "\n")
cat("Счёт:", round(best_weighted_score, 4), "\n\n")

Data[Data$ID == best_weighted_id, ]


Рекомендуемая конфигурация (по линейной свёртке): PC-0325 
Счёт: 53.1113 



,ID,CPU_Frequency_GHz,CPU_Cores,CPU_Brand,CPU_Generation,RAM_GB,Storage_GB,Storage_Type,GPU_Type,GPU_Model,⋯,Model,Purchase_Date,Age_Days,Condition,Price_USD,Power_Consumption_W,Weight_kg,Screen_Size_inch,Case_Type,Ethernet_Speed_Gbps
,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
325,PC-0325,5.01,32,Intel Core i3,7,128,2000,NVMe,Dedicated,AMD Radeon RX 7900,⋯,MacBook 3979,2022-04-06,1409,Like New,3739,444,NA,14,,10


---
# **ЧАСТЬ 2. МАСШТАБИРОВАНИЕ ИНФРАСТРУКТУРЫ**
---

## 9. Постановка задачи (Часть 2)

Во второй части рассматривается задача **двухуровневой оптимизации** инфраструктуры криптовалютной биржи (**HFT**).

Инфраструктура разбивается на три сегмента:

1. **Trading** — узлы обработки ордеров и matching engine, критична минимальная задержка.
2. **Gateways** — прием и маршрутизация сетевого трафика, важна высокая пропускная способность.
3. **DB** — хранение состояния, журналирование и доступ к данным, приоритетны RAM и Storage.

На **нижнем уровне** для каждого сегмента формируется критерий эффективности на основе нормированных характеристик серверов.

На **верхнем уровне** выполняется выбор количества серверов по сегментам с учетом ограничений по:
- числу узлов;
- стоимости инфраструктуры;
- допустимому составу сегментов.

Оптимизация проводится в два этапа:
1. сначала максимизируется интегральная эффективность инфраструктуры;
2. затем методом последовательных уступок минимизируется стоимость при допустимом снижении эффективности.

In [ ]:
# ========================================
# 1. Отбор допустимых альтернатив
# ========================================

# Оставляем серверы, удовлетворяющие минимальным требованиям
C <- Data$ID[Data$CPU_Cores >= 8 & Data$RAM_GB >= 32]
C <- na.omit(C)

cat("Количество альтернатив после первичного отбора:", length(C), "\n")


# ========================================
# 2. Формирование матрицы критериев для Парето-анализа
# ========================================

F <- data.frame(
  CPU_Frequency_GHz = Data$CPU_Frequency_GHz[Data$ID %in% C],
  CPU_Cores         = Data$CPU_Cores[Data$ID %in% C],
  RAM_GB            = Data$RAM_GB[Data$ID %in% C],
  Storage_GB        = Data$Storage_GB[Data$ID %in% C],
  Price_USD         = 1 / Data$Price_USD[Data$ID %in% C]
)

rownames(F) <- Data$ID[Data$ID %in% C]
F <- na.omit(F)

cat("Размерность матрицы критериев F:\n")
print(dim(F))
print(head(F))


# ========================================
# 3. Поиск Парето-оптимальных альтернатив
# ========================================

result <- c()

for (i in 1:nrow(F)) {
  p <- TRUE

  for (j in 1:nrow(F)) {
    if (i != j) {
      if (DPareto(F[j, ], F[i, ])) {
        p <- FALSE
        break
      }
    }
  }

  if (p) {
    result <- c(result, rownames(F)[i])
  }
}

cat("\nПарето-оптимальные альтернативы:\n")
print(result)
cat("Количество Парето-оптимальных альтернатив:", length(result), "\n")


# ========================================
# 4. Формирование таблицы Парето-оптимальных конфигураций
# ========================================

FF <- data.frame(
  CPU_Frequency_GHz = Data$CPU_Frequency_GHz[Data$ID %in% result],
  CPU_Cores         = Data$CPU_Cores[Data$ID %in% result],
  RAM_GB            = Data$RAM_GB[Data$ID %in% result],
  Storage_GB        = Data$Storage_GB[Data$ID %in% result],
  Price_USD         = Data$Price_USD[Data$ID %in% result]
)

rownames(FF) <- result

cat("\nТаблица Парето-оптимальных альтернатив FF:\n")
print(FF)


# ========================================
# 5. Нормирование критериев
# ========================================

normalize_benefit_100 <- function(x) {
  if (max(x) == min(x)) {
    return(rep(100, length(x)))
  }
  (x - min(x)) * 100 / (max(x) - min(x))
}

normalize_cost_100 <- function(x) {
  if (max(x) == min(x)) {
    return(rep(100, length(x)))
  }
  (max(x) - x) * 100 / (max(x) - min(x))
}

CPU_Cores_norm         <- normalize_benefit_100(FF$CPU_Cores)
CPU_Frequency_GHz_norm <- normalize_benefit_100(FF$CPU_Frequency_GHz)
RAM_GB_norm            <- normalize_benefit_100(FF$RAM_GB)
Storage_GB_norm        <- normalize_benefit_100(FF$Storage_GB)
Price_USD_norm         <- normalize_cost_100(FF$Price_USD)


# ========================================
# 6. Формирование нормированной таблицы
# ========================================

WW <- data.frame(
  CPU_Cores_norm,
  CPU_Frequency_GHz_norm,
  RAM_GB_norm,
  Storage_GB_norm,
  Price_USD_norm
)

rownames(WW) <- rownames(FF)

cat("\nНормированная таблица критериев WW:\n")
print(WW)

cat("\nКоличество альтернатив в WW:", nrow(WW), "\n")

Количество альтернатив после первичного отбора: 329 
Размерность матрицы критериев F:
[1] 322   5
        CPU_Frequency_GHz CPU_Cores RAM_GB Storage_GB    Price_USD
PC-0003              4.10        12    256        512 0.0003894081
PC-0006              1.74         8    128       2000 0.0004376368
PC-0007              1.34         8     32       4000 0.0004777831
PC-0008              4.65        12    128        128 0.0004089980
PC-0013              4.51        24    256        512 0.0002710027
PC-0015              1.85        10     32       1000 0.0007390983

Парето-оптимальные альтернативы:
  [1] "PC-0003" "PC-0013" "PC-0015" "PC-0021" "PC-0025" "PC-0028" "PC-0030"
  [8] "PC-0041" "PC-0046" "PC-0051" "PC-0054" "PC-0055" "PC-0063" "PC-0075"
 [15] "PC-0080" "PC-0085" "PC-0089" "PC-0092" "PC-0099" "PC-0102" "PC-0104"
 [22] "PC-0105" "PC-0108" "PC-0115" "PC-0121" "PC-0122" "PC-0123" "PC-0125"
 [29] "PC-0135" "PC-0141" "PC-0153" "PC-0154" "PC-0159" "PC-0161" "PC-0166"
 [36] "PC-0167" "PC

In [ ]:
WW

,CPU_Cores_norm,CPU_Frequency_GHz_norm,RAM_GB_norm,Storage_GB_norm,Price_USD_norm
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PC-0003,7.142857,72.50,11.111111,4.878049,87.70189
PC-0013,28.571429,82.75,11.111111,4.878049,78.50291
PC-0015,3.571429,16.25,0.000000,11.077236,97.66336
PC-0021,0.000000,60.25,49.206349,4.878049,59.44085
PC-0025,28.571429,44.25,0.000000,0.000000,92.12921
PC-0028,7.142857,50.25,49.206349,11.077236,59.91637
PC-0030,7.142857,2.25,1.587302,11.077236,95.51529
PC-0041,0.000000,10.00,11.111111,1.626016,92.05542
PC-0046,0.000000,65.50,1.587302,23.780488,94.29368


In [ ]:
FF

,CPU_Frequency_GHz,CPU_Cores,RAM_GB,Storage_GB,Price_USD
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
PC-0003,4.10,12,256,512,2568
PC-0013,4.51,24,256,512,3690
PC-0015,1.85,10,32,1000,1353
PC-0021,3.61,8,1024,512,6015
PC-0025,2.97,24,32,128,2028
PC-0028,3.21,12,1024,1000,5957
PC-0030,1.29,12,64,1000,1615
PC-0041,1.60,8,256,256,2037
PC-0046,3.82,8,64,2000,1764


In [ ]:
# ========================================
# Формирование весов сегментов и коэффициентов целевых функций
# ========================================

# 1. Относительные доли ресурсов по сегментам
cpu <- c(
  Trading  = (16 + 24) / 2,
  Gateways = (16 + 24) / 2,
  DB       = (24 + 32) / 2
)
a <- cpu / sum(cpu)

cat("Доли CPU по сегментам:\n")
print(a)


ram <- c(
  Trading  = (64 + 128) / 2,
  Gateways = (64 + 128) / 2,
  DB       = (128 + 256) / 2
)
b <- ram / sum(ram)

cat("\nДоли RAM по сегментам:\n")
print(b)


storage <- c(
  Trading  = (500 + 1000) / 2,
  Gateways = (500 + 1000) / 2,
  DB       = (2000 + 4000) / 2
)
c_share <- storage / sum(storage)

cat("\nДоли Storage по сегментам:\n")
print(c_share)


# 2. Критерии предпочтения по сегментам
# unname() нужен, чтобы не получить имена вида CPU.Trading и потом NA при обращении по "CPU"

pref_trading <- c(
  CPU = unname(a["Trading"] / (a["Trading"] + b["Trading"] + c_share["Trading"])),
  RAM = unname(b["Trading"] / (a["Trading"] + b["Trading"] + c_share["Trading"])),
  Storage = unname(c_share["Trading"] / (a["Trading"] + b["Trading"] + c_share["Trading"]))
)

pref_gateways <- c(
  CPU = unname(a["Gateways"] / (a["Gateways"] + b["Gateways"] + c_share["Gateways"])),
  RAM = unname(b["Gateways"] / (a["Gateways"] + b["Gateways"] + c_share["Gateways"])),
  Storage = unname(c_share["Gateways"] / (a["Gateways"] + b["Gateways"] + c_share["Gateways"]))
)

pref_db <- c(
  CPU = unname(a["DB"] / (a["DB"] + b["DB"] + c_share["DB"])),
  RAM = unname(b["DB"] / (a["DB"] + b["DB"] + c_share["DB"])),
  Storage = unname(c_share["DB"] / (a["DB"] + b["DB"] + c_share["DB"]))
)

cat("\nКритерии предпочтения для Trading:\n")
print(round(pref_trading, 4))

cat("\nКритерии предпочтения для Gateways:\n")
print(round(pref_gateways, 4))

cat("\nКритерии предпочтения для DB:\n")
print(round(pref_db, 4))

cat("\nИмена весов Trading:\n")
print(names(pref_trading))

cat("\nИмена весов Gateways:\n")
print(names(pref_gateways))

cat("\nИмена весов DB:\n")
print(names(pref_db))


# 3. Формализация критериев
cat("\nСформулируем критерии предпочтения для каждого сегмента:\n\n")

cat("Trading сегмент:\n")
cat("W1 = Σ(",
    round(pref_trading["CPU"], 2), "*CPUCoresNorm + ",
    round(pref_trading["RAM"], 2), "*RAMNorm + ",
    round(pref_trading["Storage"], 2), "*StorageNorm) * x_i -> max\n\n",
    sep = "")

cat("Gateways сегмент:\n")
cat("W2 = Σ(",
    round(pref_gateways["CPU"], 2), "*CPUCoresNorm + ",
    round(pref_gateways["RAM"], 2), "*RAMNorm + ",
    round(pref_gateways["Storage"], 2), "*StorageNorm) * y_i -> max\n\n",
    sep = "")

cat("DB сегмент:\n")
cat("W3 = Σ(",
    round(pref_db["CPU"], 2), "*CPUCoresNorm + ",
    round(pref_db["RAM"], 2), "*RAMNorm + ",
    round(pref_db["Storage"], 2), "*StorageNorm) * z_i -> max\n\n",
    sep = "")

cat("Критерий верхнего уровня:\n")
cat("S = Σ PriceUSD * v_i -> min\n\n")

cat("Свертка критериев нижнего уровня:\n")
cat("W = W1 + W2 + W3 -> max\n")


# 4. Проверка структуры WW
cat("\nСтолбцы таблицы WW:\n")
print(colnames(WW))

cat("\nКоличество NA в WW по столбцам:\n")
print(colSums(is.na(WW)))


# 5. Коэффициенты целевой функции стоимости
coeffS <- c(FF$Price_USD, FF$Price_USD, FF$Price_USD)

cat("\nКоэффициенты целевой функции стоимости coeffS:\n")
print(coeffS)

cat("\nДлина coeffS:\n")
print(length(coeffS))


# 6. Коэффициенты целевой функции эффективности
coeffW_trading <- pref_trading["CPU"] * WW$CPU_Cores_norm +
                  pref_trading["RAM"] * WW$RAM_GB_norm +
                  pref_trading["Storage"] * WW$Storage_GB_norm

coeffW_gateways <- pref_gateways["CPU"] * WW$CPU_Cores_norm +
                   pref_gateways["RAM"] * WW$RAM_GB_norm +
                   pref_gateways["Storage"] * WW$Storage_GB_norm

coeffW_db <- pref_db["CPU"] * WW$CPU_Cores_norm +
             pref_db["RAM"] * WW$RAM_GB_norm +
             pref_db["Storage"] * WW$Storage_GB_norm

coeffW <- c(coeffW_trading, coeffW_gateways, coeffW_db)

cat("\nКоэффициенты целевой функции эффективности coeffW:\n")
print(coeffW)

cat("\nДлина coeffW:\n")
print(length(coeffW))

cat("\nКоличество NA в coeffW:\n")
print(sum(is.na(coeffW)))

Доли CPU по сегментам:
  Trading  Gateways        DB 
0.2941176 0.2941176 0.4117647 

Доли RAM по сегментам:
 Trading Gateways       DB 
    0.25     0.25     0.50 

Доли Storage по сегментам:
  Trading  Gateways        DB 
0.1666667 0.1666667 0.6666667 

Критерии предпочтения для Trading:
    CPU     RAM Storage 
 0.4138  0.3517  0.2345 

Критерии предпочтения для Gateways:
    CPU     RAM Storage 
 0.4138  0.3517  0.2345 

Критерии предпочтения для DB:
    CPU     RAM Storage 
 0.2609  0.3168  0.4224 

Имена весов Trading:
[1] "CPU"     "RAM"     "Storage"

Имена весов Gateways:
[1] "CPU"     "RAM"     "Storage"

Имена весов DB:
[1] "CPU"     "RAM"     "Storage"

Сформулируем критерии предпочтения для каждого сегмента:

Trading сегмент:
W1 = Σ(0.41*CPUCoresNorm + 0.35*RAMNorm + 0.23*StorageNorm) * x_i -> max

Gateways сегмент:
W2 = Σ(0.41*CPUCoresNorm + 0.35*RAMNorm + 0.23*StorageNorm) * y_i -> max

DB сегмент:
W3 = Σ(0.26*CPUCoresNorm + 0.32*RAMNorm + 0.42*StorageNorm) * z_i -> max


In [ ]:
install.packages("lpSolve")
library(lpSolve)

# Количество Парето-оптимальных альтернатив
n <- nrow(FF)

cat("Количество альтернатив в одном сегменте:", n, "\n")
cat("Общее число переменных в задаче:", 3 * n, "\n")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Количество альтернатив в одном сегменте: 186 
Общее число переменных в задаче: 558 


In [ ]:
# ========================================
# Задача 1. Максимизация суммарной эффективности
# W = W1 + W2 + W3 -> max
# ========================================

objective.in<-coeffW
const.mat<-matrix(c(coeffS,
                  c(rep(1,n),rep(0,n),rep(0,n)),
                  c(rep(0,n),rep(1,n),rep(0,n)),
                  c(rep(0,n),rep(0,n),rep(1,n)),
                  c(rep(1,n),rep(0,n),rep(0,n)),
                  c(rep(0,n),rep(1,n),rep(0,n)),
                  c(rep(0,n),rep(0,n),rep(1,n))), nrow=7, byrow=TRUE)
const.dir<-c("<=",">=",">=",">=","<=","<=","<=")
const.rhs<-c(450000, 6,3,3, 10,5,5)
Res<-lp("max", objective.in, const.mat, const.dir, const.rhs, all.int=TRUE)

Res$solution

S<-function(p){return(sum(coeffS*p))}
W<-function(p){return(sum(coeffW*p))}

S(Res$solution)
W(Res$solution)

cat("Статус решения задачи W -> max:", Res_W$status, "\n")
cat("Решение:\n")
print(Res_W$solution)

[1]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 [26]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 [51]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 [76]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[101]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[126]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[151]  0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[176]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[201]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[226]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[251]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[276]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[301]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[326]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  5  0  0  0  0  0  0  0  0
[351]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[376]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[401]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[426]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[451]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[476]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[501]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[526]  0  0  5  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[551]  0  0  0  0  0  0  0  0

[1] 260520

[1] 1248.876

Статус решения задачи W -> max: 0 
Решение:
  [1]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 [26]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 [51]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
 [76]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[101]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[126]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[151]  0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[176]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[201]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[226]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[251]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
[276]  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0

In [ ]:
# ========================================
# Функции расчета критериев
# ========================================

S <- function(p) {
  return(sum(coeffS * p, na.rm = TRUE))
}

W <- function(p) {
  return(sum(coeffW * p, na.rm = TRUE))
}

# на всякий случай уберём NA из решения
sol <- Res_W$solution
sol[is.na(sol)] <- 0

cat("Значение критерия стоимости S:\n")
print(S(sol))

cat("\nЗначение критерия эффективности W:\n")
print(W(sol))

Значение критерия стоимости S:
[1] 260520

Значение критерия эффективности W:
[1] 1248.876


После решения первой задачи получаем максимальное значение интегрального критерия эффективности \(W\) при выполнении ограничений по бюджету и количеству узлов в каждом сегменте.

Далее применяется метод последовательных уступок: допускается небольшое снижение эффективности, после чего оптимизируется критерий стоимости \(S\).

In [ ]:
# ========================================
# Метод последовательных уступок
# Допускаем снижение эффективности на 5%
# ========================================

W_star <- W(Res_W$solution)
eff_drop <- 0.05
W_limit <- (1 - eff_drop) * W_star

cat("Максимальное значение эффективности W* =", W_star, "\n")
cat("Допустимое значение после уступки W_limit =", W_limit, "\n")

Максимальное значение эффективности W* = 1248.876 
Допустимое значение после уступки W_limit = 1186.432 


In [ ]:
objective.in<-coeffS
const.mat<-matrix(c(coeffW,
                  c(rep(1,n),rep(0,n),rep(0,n)),
                  c(rep(0,n),rep(1,n),rep(0,n)),
                  c(rep(0,n),rep(0,n),rep(1,n)),
                  c(rep(1,n),rep(0,n),rep(0,n)),
                  c(rep(0,n),rep(1,n),rep(0,n)),
                  c(rep(0,n),rep(0,n),rep(1,n))), nrow=7, byrow=TRUE)
const.dir<-c(">=",">=",">=",">=","<=","<=","<=")
const.rhs<-c(W_limit, 6,3,3, 10,5,5)
Res<-lp("min", objective.in, const.mat, const.dir, const.rhs, all.int=TRUE)

Res$solution

S(Res$solution)
W(Res$solution)

[1] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 [38] 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 [75] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[112] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[149] 0 0 0 0 0 0 0 9 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[186] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[223] 0 0 0 0 0 0 0 4 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[260] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[297] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[334] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0
[371] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[408] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[445] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[482] 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[519] 0 0 0 0 0 0 0 0 0 5 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
[556] 0 0 0

[1] 214602

[1] 1186.71

In [ ]:
cat("Значение критерия стоимости S после уступки:\n")
print(S(Res_S$solution))

cat("\nЗначение критерия эффективности W после уступки:\n")
print(W(Res_S$solution))

Значение критерия стоимости S после уступки:
[1] 214602

Значение критерия эффективности W после уступки:
[1] 1186.71


In [ ]:
# ========================================
# Расшифровка решения по сегментам
# ========================================

sol <- Res_S$solution

x_trading <- sol[1:n]
x_gateways <- sol[(n + 1):(2 * n)]
x_db <- sol[(2 * n + 1):(3 * n)]

RES_Trading <- data.frame(ID = rownames(FF), FF, Count = x_trading)
RES_Trading <- RES_Trading[RES_Trading$Count > 0, ]

RES_Gateways <- data.frame(ID = rownames(FF), FF, Count = x_gateways)
RES_Gateways <- RES_Gateways[RES_Gateways$Count > 0, ]

RES_DB <- data.frame(ID = rownames(FF), FF, Count = x_db)
RES_DB <- RES_DB[RES_DB$Count > 0, ]

cat("========================================\n")
cat("СЕГМЕНТ TRADING\n")
cat("========================================\n")
print(RES_Trading)

cat("\n========================================\n")
cat("СЕГМЕНТ GATEWAYS\n")
cat("========================================\n")
print(RES_Gateways)

cat("\n========================================\n")
cat("СЕГМЕНТ DB\n")
cat("========================================\n")
print(RES_DB)

СЕГМЕНТ TRADING
             ID CPU_Frequency_GHz CPU_Cores RAM_GB Storage_GB Price_USD Count
PC-0210 PC-0210              1.76        64    256       1000      5067     1
PC-0790 PC-0790              4.98         8   2048       8000     13026     9

СЕГМЕНТ GATEWAYS
             ID CPU_Frequency_GHz CPU_Cores RAM_GB Storage_GB Price_USD Count
PC-0210 PC-0210              1.76        64    256       1000      5067     4
PC-0955 PC-0955              2.82        64    512        256      6903     1

СЕГМЕНТ DB
             ID CPU_Frequency_GHz CPU_Cores RAM_GB Storage_GB Price_USD Count
PC-0790 PC-0790              4.98         8   2048       8000     13026     5


In [ ]:
# ========================================
# Итоговые характеристики по сегментам
# ========================================

calc_totals <- function(df) {
  if (nrow(df) == 0) {
    return(data.frame(
      Nodes = 0,
      CPU_Cores = 0,
      RAM_GB = 0,
      Storage_GB = 0,
      Price_USD = 0
    ))
  }

  data.frame(
    Nodes = sum(df$Count),
    CPU_Cores = sum(df$CPU_Cores * df$Count),
    RAM_GB = sum(df$RAM_GB * df$Count),
    Storage_GB = sum(df$Storage_GB * df$Count),
    Price_USD = sum(df$Price_USD * df$Count)
  )
}

tot_trading <- calc_totals(RES_Trading)
tot_gateways <- calc_totals(RES_Gateways)
tot_db <- calc_totals(RES_DB)

summary_segments <- rbind(
  cbind(Segment = "Trading", tot_trading),
  cbind(Segment = "Gateways", tot_gateways),
  cbind(Segment = "DB", tot_db)
)

cat("Итоговые характеристики по сегментам:\n")
print(summary_segments)

Итоговые характеристики по сегментам:
   Segment Nodes CPU_Cores RAM_GB Storage_GB Price_USD
1  Trading    10       136  18688      73000    122301
2 Gateways     5       320   1536       4256     27171
3       DB     5        40  10240      40000     65130


In [ ]:
# ========================================
# Общий итог по инфраструктуре
# ========================================

total_nodes <- sum(summary_segments$Nodes)
total_cpu <- sum(summary_segments$CPU_Cores)
total_ram <- sum(summary_segments$RAM_GB)
total_storage <- sum(summary_segments$Storage_GB)
total_price <- sum(summary_segments$Price_USD)

cat("========================================\n")
cat("ИТОГОВАЯ ИНФРАСТРУКТУРА HFT\n")
cat("========================================\n\n")

print(summary_segments)

cat("\n--- ОБЩИЙ ИТОГ ---\n")
cat("Всего узлов:", total_nodes, "\n")
cat("Всего CPU ядер:", total_cpu, "\n")
cat("Всего RAM (ГБ):", total_ram, "\n")
cat("Всего Storage (ГБ):", total_storage, "\n")
cat("Общая стоимость (USD):", total_price, "\n")

ИТОГОВАЯ ИНФРАСТРУКТУРА HFT

   Segment Nodes CPU_Cores RAM_GB Storage_GB Price_USD
1  Trading    10       136  18688      73000    122301
2 Gateways     5       320   1536       4256     27171
3       DB     5        40  10240      40000     65130

--- ОБЩИЙ ИТОГ ---
Всего узлов: 20 
Всего CPU ядер: 496 
Всего RAM (ГБ): 30464 
Всего Storage (ГБ): 117256 
Общая стоимость (USD): 214602 


### Вывод по второй части

Во второй части была решена задача двухуровневой оптимизации инфраструктуры криптовалютной биржи. На первом этапе максимизировался интегральный критерий эффективности \(W\), отражающий качество выбранных конфигураций в сегментах Trading, Gateways и DB. На втором этапе методом последовательных уступок минимизировалась суммарная стоимость инфраструктуры \(S\) при допустимом снижении эффективности на 5%.

В результате был получен конкретный состав инфраструктуры по сегментам, определено количество узлов каждого типа и рассчитаны итоговые суммарные характеристики системы: число узлов, общий объем CPU, RAM, Storage и полная стоимость.